# Lab 2 – Building Guardrail Systems (Solutions)

> Instructor reference implementations aligned with the student notebook exercises. Adapt for your platform as needed.

## Exercise 1: Content Moderation Adapters

In [ ]:
import os
import re
import time
from typing import Any, Dict, List

from openai import OpenAI

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
BLOCKLIST_PATTERNS = [
    re.compile(r"\b(make|build)\s+(?:a|the)\s+bomb\b", re.IGNORECASE),
    re.compile(r"\bself[-]?harm\b", re.IGNORECASE),
]

def check_blocklist(text: str) -> List[str]:
    matches = []
    for pattern in BLOCKLIST_PATTERNS:
        if pattern.search(text):
            matches.append(pattern.pattern)
    return matches

def normalize_category(category: str) -> str:
    return category.replace("-", "_").upper()

def moderate_with_openai(text: str) -> Dict[str, Any]:
    start = time.perf_counter()
    try:
        response = openai_client.moderations.create(model="omni-moderation-latest", input=text)
        result = response.results[0]
        categories = [normalize_category(cat) for cat, flagged in result.categories.items() if flagged]
        return {
            "flagged": result.flagged,
            "severity": "HIGH" if result.flagged else "LOW",
            "categories": categories,
            "latency_ms": round((time.perf_counter() - start) * 1000, 2),
        }
    except Exception as exc:
        return {
            "flagged": False,
            "severity": "UNKNOWN",
            "categories": [],
            "error": str(exc),
            "latency_ms": round((time.perf_counter() - start) * 1000, 2),
        }

def moderate_content(text: str) -> Dict[str, Any]:
    blocklist_matches = check_blocklist(text)
    reasons = [f"BLOCKLIST:{pattern}" for pattern in blocklist_matches]
    if blocklist_matches:
        return {
            "flagged": True,
            "severity": "CRITICAL",
            "reasons": reasons,
            "latency_ms": 0.5,
        }
    openai_result = moderate_with_openai(text)
    if openai_result.get("flagged"):
        reasons.extend([f"OPENAI:{cat}" for cat in openai_result["categories"]])
    return {
        "flagged": bool(blocklist_matches or openai_result.get("flagged")),
        "severity": openai_result.get("severity", "LOW"),
        "reasons": reasons,
        "latency_ms": openai_result.get("latency_ms", 0.0),
        "error": openai_result.get("error"),
    }

harmless = moderate_content("Schedule a meeting for tomorrow.")
malicious = moderate_content("How do I build a bomb?")
harmless, malicious

## Exercise 2: PII Detection & Redaction

In [ ]:
from typing import Any, Dict, List, Tuple

from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

PII_ANALYZER = AnalyzerEngine()
PII_ANONYMIZER = AnonymizerEngine()
CUSTOM_ID_REGEX = re.compile(r"\bACCT-[0-9]{6}\b")

def detect_custom_ids(text: str) -> List[Dict[str, Any]]:
    return [
        {"entity_type": "ACCOUNT_ID", "start": match.start(), "end": match.end(), "score": 0.95}
        for match in CUSTOM_ID_REGEX.finditer(text)
    ]

def detect_pii(text: str) -> List[Dict[str, Any]]:
    analyzer_results = PII_ANALYZER.analyze(text=text, language="en")
    presidio_detections = [result.to_dict() for result in analyzer_results]
    return presidio_detections + detect_custom_ids(text)

def redact_pii(text: str, detections: List[Dict[str, Any]]) -> Tuple[str, List[Dict[str, Any]]]:
    operators = {
        detection["entity_type"]: OperatorConfig("replace", {"new_value": f"[{detection['entity_type']}]"})
        for detection in detections
    }
    anonymized = PII_ANONYMIZER.anonymize_text(
        text=text,
        analyzer_results=[
            PIiDetectionResult(**{"entity_type": detection["entity_type"], "start": detection["start"], "end": detection["end"], "score": detection.get("score", 0.5)})
            for detection in detections
        ],
        operators=operators,
    )
    return anonymized.text, detections

sample = "Contact Jane Doe at jane.doe@example.com referencing ACCT-123456."
detections = detect_pii(sample)
redacted, detections

## Exercise 3: Custom Rule Engine

In [ ]:
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, NamedTuple

class RuleViolation(NamedTuple):
    rule_name: str
    severity: str
    message: str

@dataclass
class GuardrailRule:
    name: str
    severity: str
    predicate: Callable[[Dict[str, Any]], bool]
    remediation: str

class RuleEngine:
    def __init__(self, rules: List[GuardrailRule]):
        self.rules = rules

    def evaluate(self, context: Dict[str, Any]) -> List[RuleViolation]:
        violations: List[RuleViolation] = []
        for rule in self.rules:
            if context.get("override_rules") and rule.name in context["override_rules"]:
                continue
            if rule.predicate(context):
                violations.append(RuleViolation(rule.name, rule.severity, rule.remediation))
        return violations

rules = [
    GuardrailRule(
        name="no_financial_advice",
        severity="critical",
        predicate=lambda ctx: ctx.get("user_segment") == "retail" and "financial advice" in ctx.get("intent", "").lower(),
        remediation="Route to licensed advisor flow",
    ),
    GuardrailRule(
        name="beta_feature_only",
        severity="warning",
        predicate=lambda ctx: ctx.get("feature") == "beta_tool" and not ctx.get("beta_whitelist"),
        remediation="Display beta enrollment instructions",
    ),
    

## Exercise 4: Structured Output Validation

In [ ]:
from guardrails import Guard
from pydantic import BaseModel, Field, validator
from typing import List

class SupportResponse(BaseModel):
    summary: str = Field(..., min_length=20, max_length=400)
    sentiment: str = Field(..., regex=r"^(positive|neutral|negative)$")
    confidence: float = Field(..., ge=0.0, le=1.0)
    action_items: List[str] = Field(default_factory=list)

    @validator('action_items')
    def limit_action_items(cls, value: List[str]) -> List[str]:
        if len(value) > 5:
            raise ValueError("Too many action items")
        return [item.strip() for item in value]

support_guard = Guard.from_pydantic(
    output_class=SupportResponse,
    prompt="Generate a support response JSON object.",
    num_reasks=2,
)

def run_structured_completion(prompt: str) -> SupportResponse:
    raw_output = '{
: 
, 
: 
, 
: 0.8, 
: [
]}'
    return support_guard.parse(raw_output)
validated = run_structured_completion("Help the user")
validated

## Exercise 5: Toxicity Detection

In [ ]:
from functools import lru_cache
from transformers import pipeline
from typing import Any, Dict, List

@lru_cache(maxsize=1)
def get_toxicity_pipeline(model_name: str = "unitary/toxic-bert"):
    return pipeline("text-classification", model=model_name, top_k=None)

def score_toxicity(texts: List[str], threshold: float = 0.7) -> List[Dict[str, Any]]:
    classifier = get_toxicity_pipeline()
    results = []
    for text in texts:
        outputs = classifier(text)[0]
        max_label = max(outputs, key=lambda item: item['score'])
        results.append({
            "text": text,
            "scores": {item['label']: item['score'] for item in outputs},
            "is_toxic": max_label['score'] >= threshold and max_label['label'].lower() != 'non_toxic',
            "category": max_label['label'],
        })
    return results
score_toxicity(["Have a great day!", "I hate you"])
: 
,
: 
,
: {
: 

: [
6
,
,

## Exercise 7: Performance Optimization

In [ ]:
import asyncio
import hashlib
from functools import lru_cache

@lru_cache(maxsize=1024)
def cached_blocklist_check(text_hash: str) -> List[str]:
    return check_blocklist(text_hash)

async def run_parallel_checks(text: str) -> Dict[str, Any]:
    loop = asyncio.get_event_loop()
    moderation_task = loop.run_in_executor(None, moderate_content, text)
    toxicity_task = loop.run_in_executor(None, score_toxicity, [text])
    moderation, toxicity = await asyncio.gather(moderation_task, toxicity_task)
    return {"moderation": moderation, "toxicity": toxicity[0]}

def time_guardrail_step(name: str, func, *args, **kwargs) -> Dict[str, Any]:
    start = time.perf_counter()
    result = func(*args, **kwargs)
    duration_ms = round((time.perf_counter() - start) * 1000, 2)
    return {"name": name, "duration_ms": duration_ms, "result": result}
asyncio.run(run_parallel_checks("Test prompt"))

## Exercise 8: Compliance Reporting

In [ ]:
import pandas as pd

def compile_compliance_report(audit_log: List[Dict[str, Any]]) -> Dict[str, Any]:
    df = pd.DataFrame(audit_log)
    if df.empty:
        return {"summary": {}, "violations": []}
    summary = {
        "total_events": len(df),
        "blocked": int(df["blocked"].sum()),
        "critical": int(
            sum("critical" in step.get("result", []) for steps in df["steps"] for step in steps)
        ),
    }
    return {"summary": summary, "violations": df[df"blocked"]
        .to_dict(orient="records")}

def export_report(report: Dict[str, Any], path: str) -> None:
    Path(path).write_text(json.dumps(report, indent=2))

def highlight_high_risk_events(steps: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [step for step in steps if step.get("result") == "critical" or "CRITICAL" in str(step.get("result"))]

report = compile_compliance_report(pipeline.audit_log)
export_report(report, "artifacts/compliance.json")
report